In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 6** - Cake Recipie Optimisation

- This function is about optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk.
  - Each recipe is evaluated with a combined score based on _flavour_, _consistency_, _calories_, _waste_ and _cost_, where each factor contributes **negative points** as judged by an expert taster. This means the total score is negative by design.

- **Goal** - maximisation problem, your goal is to _bring that score as close to zero_ as possible or, equivalently, to maximise the negative of the total sum.

- **Goal** - maximisation function to get to 0.0

In [2]:
# Load current cumulative dataset (original + prior weekly updates) for Function 6
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_outputs.npy')

# New data for Week 6 (Function 6)
X_w5_new_point = np.array([0.592613, 0.386959, 0.657734, 0.780318, 0.073699], dtype=np.float64)
Y_w5_new_point = np.array([-0.12781660714209972], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w5_new_point.reshape(1, -1)))

# Remove duplicate X rows (if point already exists)
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w5_new_point)
Y_updated = Y_all[unique_indices]

# Save updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_6/initial_outputs.npy', Y_updated)

# Print arrays
print("Updated Inputs (X) - Function 6: ", X_unique)
print("Updated Outputs (Y) - Function 6: ", Y_updated)

Updated Inputs (X) - Function 6:  [[0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.12667892 0.2914703  0.06452848 0.6805146  0.89281919]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.441233   0.998122   0.001245   0.556711   0.321098  ]
 [0.461344   0.323423   0.803851   0.969268   0.025109  ]
 [0.487084   0.014939   0.99933    0.973696   0.099886  ]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.592613   0.386959   0.657734   0.780318   0.073699  ]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.727708   0.335856   0.785846   0.777106   0.230214  ]
 [0.7281861  0.15469257 0.73255167 0.6

### **Output Analysis**
- Last week (W4) I saw $-0.9475$, and this week (W5) I saw $-0.1278$.

- **Real-World Implication**: This is an excellent result. 
    - In the context of the cake recipe, where scores are negative based on flaws (waste, calories, cost), moving from -0.94 toward 0 indicates I have created a much more balanced recipe. 
    - I have significantly reduced the negative factors judged by the expert taster.

- **Model Clues**: This confirms that the search space is receptive to local exploitation
    - The jump from -0.94 to -0.12 suggests that I am very close to an "ideal" recipe configuration. 
    - Since the goal is to get as close to zero as possible, I am likely on the final plateau of the global maximum for this 5D function

### **Bayesian Optimisation**: Gaussian Process Regressor

- **Model Understanding**: For this 5D problem, I use a Matern kernel (nu=2.5) with ARD. ARD is vital here because in a 5D recipe, certain ingredients (like leavening agents) likely have a much tighter range of success than others (like flour or sugar).

- **Changes/Reasons**: I am maintaining the ARD structure and the `n_restarts_optimizer=25` from my Week 5 approach. 
    - The significant improvement (-0.12) proves that the model is correctly weighting the importance of the five different recipe components. 
    - Changing the model now would be counter-productive when I am so close to the target of 0.

- **Intended Impact**: To allow the GP to pinpoint the specific ingredient ratios that minimize the negative feedback. 
    - The model will now "squeeze" the uncertainty around the W5 coordinates to find the absolute peak.

In [3]:
# Define the model
kernel = C(1.0) * Matern(length_scale=[0.1]*5, nu=2.5) + WhiteKernel(noise_level=1e-5)

model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=25,
    normalize_y=True
)

#Fit the model
model.fit(X_unique, Y_updated)

/Users/prathamyeole/Library/Python/3.13/lib/python/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",1**2 * Matern...e_level=1e-05)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",25
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`G

### **Acquisition Function**

- **Function Understanding**: I am using Expected Improvement (EI) for Function 6, which is excellent for high-dimensional spaces where I need to find the absolute maximum efficiently.

- **Changes/Reasons**: I am keeping the EI strategy with xi=0.05 and the variable name improvement. 
    - This xi value provides a slight bias toward exploration, which is healthy even when I am performing well, as it ensures I don't ignore a slightly better recipe just a small distance away in 5D space.

- **Intended Impact**: To find a recipe that is statistically likely to beat my current best of -0.12. 
    - By using a dense 5D grid, I can ensure that the improvement logic covers as much of the ingredient landscape as possible.

In [4]:
def expected_improvement(X, model, y_max, xi=0.05):
    mu, sigma = model.predict(X, return_std=True)
    mu, sigma = mu.reshape(-1, 1), sigma.reshape(-1, 1)

    with np.errstate(divide='ignore'):
        improvement = mu - y_max - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma <= 0] = 0.0
    return ei.ravel()

y_max = np.max(Y_updated)

# Generate a high-density 5D random grid (100,000 points) to handle the dimensionality
x_grid = np.random.uniform(0, 1, (100000, 5))

# Calculate EI values
ei_values = expected_improvement(x_grid, model, y_max)

# Select next query
next_query = x_grid[np.argmax(ei_values)]
print(f"Strategic Week 6 Query (Function 6): [{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}]")


Strategic Week 6 Query (Function 6): [0.507314-0.394923-0.791626-0.639115-0.047392]
